# Chapter 05-08 · Bias, variance, and learning curves

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** one equation, and the two plots that make
every later decision cheaper

**Prerequisites:** 05-07 for capacity, 05-04 for MSE, 04-03 for splitting, 03-05 for variance.

**Position in the learning path:** module 05, chapter 8 of 12.

---

## Why this matters

05-07 gave you two failures with opposite fixes and a way to tell them apart after the fact. This chapter
gives you the reason they exist, and two plots that answer the questions that actually cost money:

- **"Would more data help?"** - before spending three months collecting it.
- **"Should I try a more powerful model?"** - before spending a week tuning one.

The mechanism is an exact identity. The expected squared error of a model at any point splits into
**three** parts, and only two of them are yours to control:

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\left(\mathbb{E}[\hat{f}(x)] - f(x)\right)^2}_{\text{bias}^2} + \underbrace{\text{Var}\left(\hat{f}(x)\right)}_{\text{variance}} + \underbrace{\sigma^2}_{\text{noise}}$$

**Bias** is being wrong on average - the model is too stiff to reach the truth. **Variance** is being
different every time - the model chases whatever data it was given. **Noise** is the part nobody can fix.

This chapter measures all three by simulation, confirms the identity numerically, and then shows the
learning curve that turns it into a decision.

## What you will be able to do

- State the decomposition and say which term each of 05-07's failures is
- Measure bias and variance by refitting a model on many samples
- Read a learning curve and say whether more data would help, and roughly how much
- Recognise the shape that means "you have hit the noise floor" and stop
- Explain why two models with the same error can need opposite interventions

## Warm-up: retrieve, do not reread

1. In 05-07, what happened to degree 16 as the row count went from 18 to 1,800?
2. What is the signature of underfitting in the training and held-out errors?
3. What does MSE measure, and in what units?

<br>

*Answers: (1) held-out R-squared went from -3.107 to +0.881. (2) high training error, small or negative
gap. (3) mean squared error, in the target's units squared - which is why 05-04 said to optimise it and
report RMSE.*

## Measuring the three parts

The decomposition is about an average over **datasets you did not get**. That sounds untestable, and it
is not: with a known truth you can draw four hundred datasets, fit the same model to each, and watch what
the predictions do.

- **Bias** is how far the *average* of those four hundred fits sits from the truth.
- **Variance** is how much they disagree with *each other*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


def true_function(x):
    return np.sin(1.2 * x) * 3 + 0.5 * x


NOISE_SD = 1.5
EVALUATION_GRID = np.linspace(-4, 4, 60)


def fit_one(degree, x, y):
    model = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                          StandardScaler(), LinearRegression())
    return model.fit(x.reshape(-1, 1), y)


def decompose(degree, rows_per_dataset=30, datasets=400, seed=0):
    rng = np.random.default_rng(seed)
    predictions = np.zeros((datasets, len(EVALUATION_GRID)))
    measured_errors = []
    for run in range(datasets):
        x = rng.uniform(-4, 4, rows_per_dataset)
        y = true_function(x) + rng.normal(0, NOISE_SD, rows_per_dataset)
        predictions[run] = fit_one(degree, x, y).predict(EVALUATION_GRID.reshape(-1, 1))
        fresh_y = true_function(EVALUATION_GRID) + rng.normal(0, NOISE_SD,
                                                              len(EVALUATION_GRID))
        measured_errors.append(float(((fresh_y - predictions[run]) ** 2).mean()))
    average_fit = predictions.mean(axis=0)
    return {"degree": degree,
            "bias squared": float(((average_fit - true_function(EVALUATION_GRID)) ** 2).mean()),
            "variance": float(predictions.var(axis=0).mean()),
            "noise": NOISE_SD ** 2,
            "measured error": float(np.mean(measured_errors))}


parts = pd.DataFrame([decompose(degree) for degree in [1, 2, 3, 4, 5, 6, 7, 8]])
parts["bias2 + var + noise"] = (parts["bias squared"] + parts["variance"] + parts["noise"])
parts["gap"] = parts["measured error"] - parts["bias2 + var + noise"]
print(parts.to_string(index=False, float_format=lambda v: "%.4f" % v))

**The last column is the point: the identity holds.** Every row's measured error matches
`bias² + variance + noise` to within about a tenth, on numbers ranging from 7 to 49. The decomposition is
not a metaphor - it is an equation, and this is it being checked.

**Now read the two middle columns down the table.**

| degree | bias² | variance |
|---|---|---|
| 1 | **4.5136** | 0.4178 |
| 3 | 1.1162 | 0.7281 |
| **5** | **0.0542** | **1.2954** |
| 8 | 0.0284 | **46.7088** |

**Bias falls and variance rises, and they cross.** Degree 1 is wrong on average by a lot and wrong the
*same* way every time. Degree 8 is right on average and never twice the same. The best total error is at
degree 5, where bias has almost vanished and variance has not yet exploded.

**Degree 2 is worth a glance too:** its bias² is *higher* than degree 1's, 4.5439 against 4.5136. Adding
capacity does not have to reduce bias - a squared term is no help in approximating a function that is
mostly a sine wave, and it costs variance regardless.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)

for ax, degree in zip(axes, [1, 5, 8]):
    rng = np.random.default_rng(0)
    collected = []
    for run in range(40):
        x = rng.uniform(-4, 4, 30)
        y = true_function(x) + rng.normal(0, NOISE_SD, 30)
        collected.append(fit_one(degree, x, y).predict(EVALUATION_GRID.reshape(-1, 1)))
    collected = np.array(collected)

    for single in collected[:25]:
        ax.plot(EVALUATION_GRID, single, color="#0072B2", alpha=0.16, linewidth=1.2)
    ax.plot(EVALUATION_GRID, collected.mean(axis=0), color="#D55E00", linewidth=3,
            label="the average fit")
    ax.plot(EVALUATION_GRID, true_function(EVALUATION_GRID), color="#000000",
            linewidth=2.6, linestyle="--", label="the truth")
    row = parts[parts.degree == degree].iloc[0]
    ax.set_title("degree %d\nbias%s %.2f    variance %.2f"
                 % (degree, chr(178), row["bias squared"], row["variance"]), fontsize=11)
    ax.set_xlabel("x")
    ax.legend(fontsize=8.5, loc="upper left")

axes[0].set_ylim(-12, 12)
axes[0].set_ylabel("prediction")
plt.tight_layout()
plt.show()

**Each faint blue line is one model, fitted to thirty rows drawn afresh. This picture is the equation.**

- **Degree 1: the blue lines almost coincide, and the orange average sits well away from the black
  truth.** That is bias - a stiff model gives you nearly the same answer whatever data you hand it, and
  that answer is wrong. **You could average a thousand of these and never get closer.**
- **Degree 5: the average sits on the truth, and the individual lines are close to it.** Bias 0.05,
  variance 1.30.
- **Degree 8: the average is still on the truth, and the individual lines are everywhere**, especially at
  the edges. Any one of them is a model somebody would have shipped.

> **Bias is the distance from the orange line to the black one. Variance is the width of the blue
> spray.**

**And the asymmetry that matters in practice:** bias cannot be averaged away, but variance can. Fit a
high-variance model twenty times on different samples and average the predictions, and the spray narrows
while the average stays on the truth. **That is exactly what a random forest is** - 05-10 builds one -
and it is why the trick works on high-variance models and does nothing for high-bias ones.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

left.plot(parts.degree, parts["bias squared"], "o-", color="#0072B2", linewidth=2.4,
          markersize=8, label="bias squared")
left.plot(parts.degree, parts["variance"], "s-", color="#D55E00", linewidth=2.4,
          markersize=8, label="variance")
left.plot(parts.degree, parts["measured error"], "^-", color="#7B3294", linewidth=2.4,
          markersize=8, label="total measured error")
left.axhline(NOISE_SD ** 2, color="#000000", linestyle="--", linewidth=1.6,
             label="noise floor, %.2f" % NOISE_SD ** 2)
best = int(parts.loc[parts["measured error"].idxmin(), "degree"])
left.axvline(best, color="#009E73", linewidth=2, alpha=0.5)
left.set_yscale("log")
left.set_xlabel("polynomial degree")
left.set_ylabel("contribution to squared error (log scale)")
left.set_title("Bias falls, variance rises, and they cross", fontsize=11)
left.legend(fontsize=8.5)

bottom = np.zeros(len(parts))
for column, colour in [("bias squared", "#0072B2"), ("variance", "#D55E00"),
                       ("noise", "#BBBBBB")]:
    right.bar(parts.degree, parts[column], bottom=bottom, color=colour, label=column,
              width=0.7)
    bottom = bottom + parts[column].to_numpy()
right.set_ylim(0, 12)
right.set_xlabel("polynomial degree")
right.set_ylabel("squared error")
right.set_title("The same thing stacked (degrees 7-8 run off the top)", fontsize=11)
right.legend(fontsize=9)

plt.tight_layout()
plt.show()

**The grey band never moves.** Whatever you do to the model, 2.25 of the squared error is the noise the
data was generated with, and no algorithm can go below it.

**That band is the most useful thing on the plot**, because it is the answer to "how good could this
get?". A model sitting at 3.60 when the floor is 2.25 has about 1.35 of addressable error left; a model
at 2.30 is finished, and further work is wasted whatever the business would like the number to be.

**The catch is that in real work you do not know the noise floor.** Three ways to estimate it, in
descending order of reliability:

1. **Duplicate measurements.** If the same unit was measured twice, the spread between the pairs *is* the
   noise, and nothing else.
2. **The best model anyone has managed**, including a much more flexible one. If gradient boosting,
   a neural network and a linear model all stop at the same error, that is strong evidence for a floor.
3. **Domain knowledge.** "Delivery times are unpredictable to within about ten minutes because of
   traffic" is a floor, stated by someone who knows.

## The learning curve

The decomposition explains *why* a model is failing. The **learning curve** answers what to do about it,
and it is the highest-value plot in this module.

The idea is simple: **fit the same model on 20 rows, then 40, then 80, and so on, and track two errors** -
the error on the rows it was fitted to, and the error on rows it was not. Then read the shape.

In [ ]:
from sklearn.model_selection import KFold, learning_curve

# SYNTHETIC: 1,200 rows from the same generator, so the noise floor is known to be 1.5
curve_rng = np.random.default_rng(11)
n_available = 1200
all_x = curve_rng.uniform(-4, 4, n_available)
all_y = true_function(all_x) + curve_rng.normal(0, NOISE_SD, n_available)

sizes = np.array([20, 40, 80, 160, 320, 640])


def learning_curve_for(degree):
    pipeline = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                             StandardScaler(), LinearRegression())
    counts, train_scores, validation_scores = learning_curve(
        pipeline, all_x.reshape(-1, 1), all_y, train_sizes=sizes,
        cv=KFold(5, shuffle=True, random_state=0),
        scoring="neg_root_mean_squared_error")
    return counts, -train_scores.mean(axis=1), -validation_scores.mean(axis=1)


for degree, label in [(1, "degree 1"), (5, "degree 5"), (14, "degree 14")]:
    counts, train_line, validation_line = learning_curve_for(degree)
    print("%-10s train %s" % (label, np.round(train_line, 3)))
    print("%-10s valid %s" % ("", np.round(validation_line, 3)))
    print("%-10s gap   %s\n" % ("", np.round(validation_line - train_line, 3)))
print("the noise floor is RMSE %.2f" % NOISE_SD)

### Predict before running

Three models, three shapes. Before the plot: which of the three would benefit from another 600 rows?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)

for ax, (degree, title) in zip(axes, [(1, "degree 1: high bias"),
                                      (5, "degree 5: about right"),
                                      (14, "degree 14: high variance")]):
    counts, train_line, validation_line = learning_curve_for(degree)
    ax.plot(counts, train_line, "o-", color="#0072B2", linewidth=2.4, markersize=8,
            label="on the rows it was fitted to")
    ax.plot(counts, validation_line, "s-", color="#D55E00", linewidth=2.4, markersize=8,
            label="on rows it has not seen")
    ax.fill_between(counts, train_line, validation_line, color="#D55E00", alpha=0.12)
    ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.8,
               label="noise floor, %.2f" % NOISE_SD)
    ax.set_xscale("log")
    ax.set_xticks(sizes)
    ax.set_xticklabels([str(size) for size in sizes], fontsize=9)
    ax.minorticks_off()
    ax.set_ylim(0.8, 4.2)
    ax.set_xlabel("rows used to fit (log scale)")
    ax.set_title("%s\nfinal gap %.3f" % (title, validation_line[-1] - train_line[-1]),
                 fontsize=11)
    ax.legend(fontsize=8, loc="upper right")

axes[0].set_ylabel("RMSE")
plt.tight_layout()
plt.show()

**Three shapes, three different decisions, and none of them requires knowing what the model is.**

**Degree 1 - the curves meet, and they meet in the wrong place.** Validation falls from 2.688 to 2.616
and stops; training rises from 2.511 to 2.576; the final gap is **0.040**. They have converged at 2.6
against a floor of 1.50.

> **More data will not help. The model has learned everything these rows can teach it, and it is still
> 1.1 RMSE above the floor.** That excess is bias, and the fix is a better model.

**Degree 5 - the curves meet at the floor.** Validation reaches 1.523, training 1.509, gap **0.015**,
floor 1.50. There is nothing left to get. **Stop.**

**Degree 14 - the gap is enormous and closing fast.** Validation starts at **405.4** on twenty rows - a
polynomial with fourteen terms fitted to twenty points, extrapolating wildly - and reaches **1.522** at
640 rows. The final gap is 0.037.

> **More data helped, and it did the entire job.** At 640 rows degree 14 is indistinguishable from degree
> 5 (1.522 against 1.523). This is 05-07's finding as a curve: **overfitting is a shortage of rows, not a
> property of the model.**

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.set_xlim(0, 10.4)
ax.set_ylim(0, 5.2)
ax.axis("off")

boxes = [
    (0.2, 3.5, "#F4CCCC", "The curves have MET,\nwell above the floor",
     "bias. More data is wasted.\nMore capacity, better features,\na different model class."),
    (3.6, 3.5, "#FCE5CD", "A LARGE gap, still closing",
     "variance. More data helps,\nand the curve says roughly\nhow much you still need."),
    (7.0, 3.5, "#D9EAD3", "The curves have MET,\nat the floor",
     "nothing left. Stop.\nAny further work is spent\non irreducible noise."),
]
for x0, y0, colour, heading, action in boxes:
    ax.add_patch(plt.Rectangle((x0, y0), 3.2, 1.5, facecolor=colour, edgecolor="#666666",
                               linewidth=1.4))
    ax.text(x0 + 1.6, y0 + 1.05, heading, fontsize=10.5, fontweight="bold", ha="center")
    ax.text(x0 + 0.15, y0 + 0.42, action, fontsize=9, va="center")

ax.text(5.2, 5.0, "READING A LEARNING CURVE", fontsize=13, fontweight="bold", ha="center")

ax.text(0.2, 2.9, "AND THE ONE THAT IS NOT A SHAPE:", fontsize=10.5, fontweight="bold")
ax.text(0.2, 2.45, "The validation curve turns UPWARD as rows are added.", fontsize=10)
ax.text(0.2, 2.05, "That is not a bias-variance story - it is a bug. Usually the extra rows",
        fontsize=9.5, color="#444444")
ax.text(0.2, 1.7, "come from a different period, source or population than the first ones.",
        fontsize=9.5, color="#444444")

ax.text(0.2, 1.05, "WHAT THE GAP MEANS", fontsize=10.5, fontweight="bold")
ax.text(0.2, 0.6, "gap = variance      distance from the meeting point to the floor = bias",
        fontsize=10)
ax.text(0.2, 0.2, "so a learning curve shows you both terms of the decomposition at once.",
        fontsize=9.5, color="#444444")

plt.tight_layout()
plt.show()

## Turning the curve into a number: how many more rows?

"More data would help" is not yet a decision. The curve can be extrapolated, and the shape it follows is
usually close to a power law: **the excess error above the floor falls as some power of the row count.**

In [ ]:
counts, _, validation_line = learning_curve_for(14)
usable = counts >= 80                                    # skip the wild small-n points

slope, intercept = np.polyfit(np.log(counts[usable]),
                              np.log(validation_line[usable] - NOISE_SD), 1)
print("excess RMSE above the floor  =  %.3f x n^(%.3f)\n" % (np.exp(intercept), slope))

for target_rows in [640, 1200, 2400, 5000]:
    predicted = NOISE_SD + np.exp(intercept) * target_rows ** slope
    print("  at %5d rows, predicted validation RMSE %.4f" % (target_rows, predicted))
print("\n  observed at 640 rows: %.4f" % validation_line[-1])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))

extended = np.geomspace(60, 8000, 200)
ax.plot(extended, NOISE_SD + np.exp(intercept) * extended ** slope, color="#7B3294",
        linewidth=2.2, linestyle="--", label="the fitted power law, extrapolated")
ax.plot(counts[usable], validation_line[usable], "o", color="#009E73", markersize=10,
        label="measured, and used for the trend")
ax.text(24, 2.25, "n = 20 and 40 measured\n405.4 and 3.76 - off the top,\nand excluded from the fit",
        fontsize=9, color="#D55E00")
ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.8,
           label="noise floor, %.2f" % NOISE_SD)
for target_rows in [1200, 2400, 5000]:
    predicted = NOISE_SD + np.exp(intercept) * target_rows ** slope
    ax.plot([target_rows], [predicted], "*", color="#7B3294", markersize=15)
    ax.annotate("%.3f" % predicted, (target_rows, predicted), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=9.5)

ax.set_xscale("log")
ax.set_ylim(1.45, 2.4)
ax.set_xlabel("rows used to fit (log scale)")
ax.set_ylabel("validation RMSE")
ax.set_title("Eight times the data buys 0.019 RMSE. That is a decision, not an opinion.",
             fontsize=11.5)
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

**Doubling from 640 to 1,200 rows buys 0.011 RMSE. Going to 5,000 - nearly eight times the data -
buys 0.019.**

That is a decision, and it is a clear no. The model is 0.022 above a floor of 1.50, so **there is almost
nothing left to buy at any price** - and now that is a sentence with numbers behind it rather than an
opinion.

**Two cautions about extrapolating.**

The small-`n` points were excluded deliberately. At 20 rows the validation RMSE was 405, and including it
would let one wild fit dominate the regression. Fit the trend on the part of the curve that is already
behaving.

**And the extrapolation assumes the new rows look like the old ones.** They frequently do not, which is
the next section.

## Failure lab: the curve that goes the wrong way

Every learning curve so far went down. Here is one that does not.

In [ ]:
# SYNTHETIC: the first 300 rows come from the original process. Everything after
# row 300 comes from a "new supplier" whose target is scaled and shifted.
supplier_rng = np.random.default_rng(23)
CLEAN_ROWS = 300
total_rows = 2000

supply_x = supplier_rng.uniform(-4, 4, total_rows)
supply_y = true_function(supply_x) + supplier_rng.normal(0, NOISE_SD, total_rows)

from_new_supplier = np.arange(total_rows) >= CLEAN_ROWS
supply_y[from_new_supplier] = (0.7 * true_function(supply_x[from_new_supplier]) + 4.0
                               + supplier_rng.normal(0, NOISE_SD,
                                                     from_new_supplier.sum()))

check_x = supplier_rng.uniform(-4, 4, 600)
check_y = true_function(check_x) + supplier_rng.normal(0, NOISE_SD, 600)

drift_table = []
for used in [100, 200, 300, 500, 800, 1200, 2000]:
    model = fit_one(5, supply_x[:used], supply_y[:used])
    error = float(np.sqrt(((check_y - model.predict(check_x.reshape(-1, 1))) ** 2).mean()))
    drift_table.append({"rows used": used,
                        "of which from the new supplier": max(0, used - CLEAN_ROWS),
                        "validation RMSE": error})
print(pd.DataFrame(drift_table).to_string(index=False, float_format=lambda v: "%.4f" % v))
print("\nthe noise floor is %.2f" % NOISE_SD)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
frame = pd.DataFrame(drift_table)
ax.plot(frame["rows used"], frame["validation RMSE"], "o-", color="#D55E00",
        linewidth=2.6, markersize=10)
ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.8,
           label="noise floor, %.2f" % NOISE_SD)
ax.axvline(CLEAN_ROWS, color="#0072B2", linewidth=2, linestyle=":",
           label="the new supplier starts here")
ax.annotate("adding 1,700 rows\nmade it 2.4x worse",
            xy=(2000, frame["validation RMSE"].iloc[-1]), xytext=(700, 3.6),
            arrowprops=dict(arrowstyle="->", linewidth=1.8, color="#000000"),
            fontsize=10.5, fontweight="bold")
ax.set_xlabel("rows used to fit")
ax.set_ylabel("validation RMSE on clean rows")
ax.set_title("A learning curve that goes up is not a bias-variance story", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The model gets steadily worse as data is added: 1.60 at 300 rows, 3.87 at 2,000.**

Nothing in the bias-variance decomposition predicts this, because the decomposition assumes every row
comes from the same distribution. **Here they do not**, and the extra rows are teaching the model a
relationship that does not hold on the rows it will be scored on.

> **An upward-sloping validation curve is a data problem, not a modelling problem.** No choice of degree,
> penalty or algorithm fixes it.

**What causes it in practice** - all of these are common:

- **A source or vendor changed** partway through the history, as here.
- **The backfill is different from the live feed.** Historical rows were reconstructed, and the
  reconstruction is not what the pipeline produces today.
- **A definition changed** - the target was recorded differently before some date.
- **The new rows are a different population** - a new market, a new customer segment, a new device.

**What to do:** find the boundary and split on it. Here, training on the first 300 rows alone gives
**1.600** against 3.867 with all 2,000 - so the entire fix is to *discard* 1,700 rows, which is a
conclusion no amount of model tuning would ever reach.

**And the check that catches it early:** add a column for the source or the date and see whether the
model uses it. If "which batch is this row from" predicts the target, the batches are not
interchangeable - which is 04-04's grouped-split reasoning, arriving from the learning curve.

## Common misconceptions

**"Bias means the model is unfair."**
Different word, different field. Here bias is a statistical quantity: how far the *average* fit sits from
the truth. Algorithmic fairness is a real and separate subject, covered in module 13.

**"High variance means the predictions are noisy."**
It means they change a lot **when the training data changes**. A high-variance model is perfectly
deterministic - run it twice on the same data and you get the same answer. The variation is across
datasets you did not get, which is why it takes a simulation to see and a held-out set to detect.

**"Adding capacity reduces bias."**
Usually, not always. Degree 2 had *higher* bias² than degree 1 here - 4.5439 against 4.5136 - because a
squared term is no use in approximating a sine. Capacity only reduces bias if it is capacity of the right
shape, and it costs variance either way.

**"More data always helps."**
It helps with variance and does nothing for bias. Degree 1's curves had already met at 2.6 with a floor
of 1.50, and the next ten thousand rows would have changed nothing. And when the new rows come from a
different process, more data actively hurts - 1.60 to 3.87 in the failure lab.

**"The gap between training and validation error is the thing to minimise."**
Then fit a constant: gap zero, error terrible. Degree 1 had a final gap of 0.040 and was the worst model
of the three. **The gap is variance; the distance from the meeting point to the floor is bias.** Both
matter and they are read from different parts of the plot.

**"I should pick the model with the lowest bias."**
You should pick the lowest **total**. Degree 8 had the lowest bias² of the sweep, 0.0284, and a total
error of 48.98 against degree 5's 3.60.

**"A learning curve needs a lot of data to draw."**
It needs the data you already have. Every point on it is fitted on a *subset* of what you hold, which is
what makes it answer "would more help?" without collecting any more.

## Exercises

Solutions: `solutions/05_regression/05-08_bias_variance_solutions.ipynb`.

### Quick understanding

**E1.** Write the three-way decomposition and say, in one clause each, what the terms mean.

**E2.** Which term does more data reduce, and which does more capacity reduce?

**E3.** Why can bias not be reduced by averaging many fits, while variance can?

### Hand calculation

**E4.** A model is fitted on five different samples and predicts 12, 14, 11, 15, 13 at a point where the
truth is 10. Compute the bias, the bias squared, and the variance at that point.

**E5.** For the same five predictions, suppose the noise variance is 4. Give the expected squared error
at that point, and say which term dominates.

**E6.** Model A has bias² 9 and variance 1; model B has bias² 1 and variance 9. The noise variance is 4.
Which do you prefer, and what would change your answer?

**E7.** A learning curve shows training RMSE 3.0 and validation RMSE 3.1 at 1,000 rows, and the noise
floor is believed to be 1.0. Diagnose it and give your next action.

### Coding

**E8.** Write `bias_variance(model_factory, truth_fn, n_rows, datasets)` returning the three terms, and
verify the identity holds on two models of your choice.

**E9.** Reproduce the bias-variance sweep with `n_rows=200` instead of 30, and report how the crossover
degree moves. Explain the direction of the shift.

**E10.** Draw a learning curve for a model of your choice on the California Housing data from 05-05, and
state whether you would collect more block groups.

**E11.** Show that averaging 25 independently-fitted degree-8 models reduces the variance term, and
report by roughly what factor. Compare with what happens to the bias.

**E12.** Modify the failure lab so that the new supplier's rows are merely *noisier* rather than shifted.
Does the learning curve still turn upward? Explain the difference.

### Interpretation

**E13.** Your learning curve's two lines are still far apart at the largest sample size you can plot, and
both are still falling. What do you report to a manager asking whether to fund more data collection?

**E14.** Two teams report the same validation RMSE on the same data. One has a training RMSE far below
it, the other's is nearly equal. Say what each should do next.

### Debugging

**E15.** Your learning curve is jagged - validation error jumps around by 20% between adjacent sample
sizes. Give the cause and the fix.

**E16.** A learning curve computed with `cv=5` looks fine, and the model fails in production. Name three
things the curve could not have told you.

### Exam and interview reasoning

**E17.** "Explain the bias-variance tradeoff." Answer in under a minute, then handle: "modern deep
networks have enormous capacity and low bias *and* low variance - does that break it?"

### Transfer to a different situation

**E18.** You have 2,000 labelled rows, labelling costs 8 euros per row, and you have a budget for 3,000
more. Describe exactly what you would compute before spending any of it.

### Explain it to someone non-technical

**E19.** In under 90 words, explain bias and variance using something other than models or data.

### Optional challenge

**E20.** Derive the decomposition algebraically from `E[(y - f_hat)²]`, stating where the assumption that
the noise is independent of the model is used.

**E21.** Show empirically that the variance term scales roughly as `p / n` for a linear model with `p`
parameters fitted on `n` rows, by measuring it across a grid of both.

## Mastery check

- [ ] State the decomposition and identify each term in a described situation
- [ ] Measure bias and variance by simulation when the truth is known
- [ ] Read the three learning-curve shapes and give the right action for each
- [ ] Recognise an upward-sloping validation curve as a data problem
- [ ] Extrapolate a curve to answer "how many more rows?" with a number
- [ ] Explain why averaging models works on variance and not on bias

## What should now feel instinctive

- Drawing a learning curve before proposing to collect more data
- Asking "where is the noise floor?" of any model that is not good enough
- Reading a small train/validation gap as a question, not a success
- Treating "more data" and "more capacity" as answers to different diagnoses
- Suspecting the data source when a curve moves the wrong way

## Flashcards

| Front | Back |
|---|---|
| The decomposition | expected squared error = bias² + variance + noise |
| Bias | how far the *average* fit is from the truth - a stiff model, wrong the same way each time |
| Variance | how much fits disagree with *each other* across training samples |
| Noise | irreducible. 2.25 here, by construction |
| Degree 1 vs degree 8 here | bias² 4.51 / var 0.42, against bias² 0.03 / var 46.71 |
| Best total | degree 5: bias² 0.054, variance 1.295, total 3.60 |
| Curves meet above the floor | bias. More data is wasted; change the model |
| Curves still far apart | variance. More data helps |
| Curves meet at the floor | done. Stop |
| Curve slopes upward | a data problem - a source, definition or population changed |
| The failure lab | 1.60 at 300 rows, 3.87 at 2,000. The fix was discarding 1,700 rows |
| Averaging many fits | cuts variance, leaves bias untouched - which is what a random forest is |

## Next

**05-09 · Regularisation: ridge, lasso, and the coefficient path.** 05-07 found that overfitting looks
like enormous, mutually-cancelling coefficients - 5.07 at degree 1 growing to 8,331 at degree 14. This
chapter has now named that as variance.

The next chapter attacks it directly: keep every column, and add a penalty on coefficient size to the
loss. That converts "which features do I drop?" - a discrete, awkward question - into "how hard do I
push?", a single continuous dial that cross-validation can turn for you.